# 26. LLM-as-Judge

**Tier:** Evaluation & Production
**Estimated time:** 50 minutes
**Prerequisites:** 23, 24
**Priority:** 🔴 Crucial — at scale, judges are how *everything* gets scored, and an uncalibrated judge silently ratifies garbage; knowing the bias catalog is table stakes for anyone shipping evals. *If skipped, revisit when:* n/a — pairs directly with notebook 24's model-graded scorer; skipping it means your evals inherit undetected bias.
**Source material:** Stanford Lecture 8 (LLM evaluation, biases, pitfalls) — https://x.com/ajitcodes/status/2057043965317165490

## What You'll Learn
- The three judge biases that show up constantly in practice: position, verbosity, self-preference
- Pairwise comparison vs. rubric (absolute) scoring — and why they fail differently
- How to calibrate a judge against a small human-labeled set before trusting it
- When NOT to use a judge at all

## Why This Matters
Notebook 24 introduced the model-graded scorer as the most flexible option. It's also the easiest to fool yourself with — an uncalibrated judge can rate a worse answer higher just because it's longer, or because it agrees with itself, and you won't notice unless you deliberately test for it. This notebook connects directly to notebook 23's "agentic laziness / self-preferential bias" failure modes and to notebook 27b's agent-eval trajectory scoring, both of which lean on a judge you can trust.


## Why judges go wrong

An LLM judge is just another LLM call — it inherits every bias a language model has, plus a few specific to the grading task itself:

- **Position bias** — in a pairwise "which is better, A or B?" comparison, judges systematically favor whichever answer appears *first* in the prompt, independent of content. Swap the order and the verdict can flip.
- **Verbosity bias** — longer answers get rated as "more thorough" or "more helpful" even when they're not more correct — padding looks like effort.
- **Self-preference bias** — a model asked to judge outputs (including its own) tends to rate outputs in its own style more favorably, even when a different style is equally correct (this is the same "self-preferential bias" failure mode from notebook 23's solo-prompting problems, now showing up in the grader instead of the generator).

We'll demonstrate all three, then build defenses for each.

In [ ]:
import os, re
import numpy as np

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — judge demo cells will be skipped.")

def ask(prompt, system="You are concise.", max_tokens=300, temperature=0.7):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]"
    try:
        msg = client.messages.create(model=TEACH_MODEL, max_tokens=max_tokens, system=system,
                                      temperature=temperature, messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    except Exception as e:
        return f"[skipped: {type(e).__name__}: {str(e)[:120]}]"


## Demo 1 — position bias

We'll ask a judge to pick the better of two *equally good* answers, then run it twice with the order swapped. If the verdict changes just from reordering, that's position bias — not a real quality difference.

In [ ]:
PAIRWISE_JUDGE_SYSTEM = (
    "You are comparing two answers to the same question. Reply with ONLY 'A' or 'B' "
    "for whichever is better. No explanation."
)

ANSWER_1 = "The mitochondria is the organelle that generates most of a cell's ATP through respiration."
ANSWER_2 = "Mitochondria generate ATP for the cell via cellular respiration; they're often called the powerhouse of the cell."

def pairwise_judge(question, ans_a, ans_b):
    prompt = f"Question: {question}\nAnswer A: {ans_a}\nAnswer B: {ans_b}\nWhich is better?"
    return ask(prompt, system=PAIRWISE_JUDGE_SYSTEM, max_tokens=5, temperature=0.0).strip()

question = "What do mitochondria do in a cell?"
verdict_order1 = pairwise_judge(question, ANSWER_1, ANSWER_2)   # ANSWER_1 = A, ANSWER_2 = B
verdict_order2 = pairwise_judge(question, ANSWER_2, ANSWER_1)   # swapped: ANSWER_2 = A, ANSWER_1 = B

print(f"Order 1 (1=A, 2=B): judge picked {verdict_order1}")
print(f"Order 2 (2=A, 1=B): judge picked {verdict_order2}")
if HAS_ANTHROPIC:
    flipped = (verdict_order1 == "A") != (verdict_order2 == "B")
    print("Position bias detected: the SAME answer won regardless of slot." if not flipped
          else "No position bias detected in this run (the winning ANSWER was consistent).")


## Demo 2 — verbosity bias

Same fact, two lengths — one terse, one padded with filler that adds no new information. A verbosity-biased judge rates the padded one higher purely on length.

In [ ]:
TERSE_ANSWER = "Paris."
PADDED_ANSWER = (
    "That's a great question! Let me think about this carefully. Geographically speaking, "
    "when we consider the countries of Europe and their administrative centers, the answer "
    "you're looking for, after careful consideration, is Paris."
)

RUBRIC_JUDGE_SYSTEM = (
    "Rate how good this answer is on a scale of 0 to 10, considering only correctness and "
    "helpfulness. Reply with ONLY the number."
)

def rubric_judge(question, answer):
    prompt = f"Question: {question}\nAnswer: {answer}\nRating (0-10):"
    raw = ask(prompt, system=RUBRIC_JUDGE_SYSTEM, max_tokens=5, temperature=0.0)
    try:
        return float(re.findall(r"\d+(?:\.\d+)?", raw)[0])
    except (IndexError, ValueError):
        return None

q = "What is the capital of France?"
terse_rating = rubric_judge(q, TERSE_ANSWER)
padded_rating = rubric_judge(q, PADDED_ANSWER)
print(f"Terse answer ({len(TERSE_ANSWER.split())} words) rated: {terse_rating}")
print(f"Padded answer ({len(PADDED_ANSWER.split())} words) rated: {padded_rating}")


## Demo 3 — self-preference bias

We generate an answer with our teach model, then ask that SAME model to judge its own answer against a stylistically different (but equally correct) alternative. Self-preference bias shows up as the judge favoring its own phrasing style — the same failure mode notebook 23 calls out for solo agent loops, here relocated into the grading step.

In [ ]:
OWN_STYLE_ANSWER = ask(
    "In one sentence, explain what a hash map is.", max_tokens=60, temperature=0.0
)
ALT_STYLE_ANSWER = (
    "hash map = data structure. key -> bucket via hash function -> O(1) avg lookup/insert."
)

verdict = pairwise_judge(
    "Explain what a hash map is.", OWN_STYLE_ANSWER, ALT_STYLE_ANSWER
)
print(f"Own-style answer: {OWN_STYLE_ANSWER!r}")
print(f"Alt-style answer: {ALT_STYLE_ANSWER!r}")
print(f"Judge (same model) picked: {verdict}")
print("Both answers are factually correct — a verdict for 'A' (its own style) on every")
print("re-run regardless of content is the self-preference signature to watch for.")


## Pairwise vs. rubric (absolute) scoring

Two different judge protocols, with different failure modes:

- **Pairwise** ("which is better, A or B?") is more reliable for *ranking* two close alternatives, but suffers from position bias (Demo 1) and doesn't give you an absolute quality number — you can't compare pairwise results across different pairs of candidates.
- **Rubric / absolute scoring** ("rate 0-10 against these criteria") gives a number you can track over time and compare across different eval runs, but is more vulnerable to verbosity bias (Demo 2) and to drifting criteria interpretation across a large eval set.

A durable pattern: use rubric scoring for tracking trend lines over time (feeds notebook 27's monitoring dashboards), and pairwise only for a final, low-volume "which do we ship" decision — with the debiasing defenses below applied first.

## Defenses

1. **Randomize position, then average.** Run the pairwise comparison in both orders and only trust a verdict that agrees both ways (exactly what Demo 1 does above) — an even simpler fix than an explicit debias formula.
2. **Judge on a length-normalized rubric.** Make brevity an explicit criterion ("penalize unnecessary padding") instead of hoping the judge infers it, and cap `max_tokens` on candidate answers so verbosity differences shrink before they reach the judge.
3. **Use a different, ideally stronger, model as judge than the model being judged**, and never let a model self-report on its own past-conversation output without a second reference point.
4. **Calibrate against a small human-labeled set.** Before trusting a judge at all, run it on ~10-20 examples you've scored by hand and check agreement — the next cell builds this.

In [ ]:
# Calibration: compare judge scores to a small human-labeled set.
HUMAN_LABELED = [
    {"q": "What is 2 + 2?", "a": "4", "human_score": 10},
    {"q": "What is 2 + 2?", "a": "I think it might be 5, not fully sure.", "human_score": 1},
    {"q": "What is the capital of Italy?", "a": "Rome", "human_score": 10},
    {"q": "What is the capital of Italy?", "a": "Milan", "human_score": 0},
]

judge_scores = [rubric_judge(item["q"], item["a"]) for item in HUMAN_LABELED]
human_scores = [item["human_score"] for item in HUMAN_LABELED]

valid = [(j, h) for j, h in zip(judge_scores, human_scores) if j is not None]
if valid:
    j_arr, h_arr = zip(*valid)
    correlation = np.corrcoef(j_arr, h_arr)[0, 1]
    print(f"Judge vs. human correlation: {correlation:.2f}  (>0.7 is a reasonable bar to trust the judge)")
    for item, j in zip(HUMAN_LABELED, judge_scores):
        print(f"  human={item['human_score']:>2}  judge={j}   {item['a'][:40]!r}")
else:
    print("[skipped: no ANTHROPIC_API_KEY]")


## When NOT to use a judge

A judge is the wrong tool when:

- **An exact or rubric scorer already exists and is unambiguous** (notebook 24) — don't spend a model call grading something a regex can grade for free and more reliably.
- **The stakes are high and errors are asymmetric** (medical, legal, financial claims) — a judge's mistakes are themselves unverified; use human review as the final gate.
- **You haven't calibrated it yet** — an uncalibrated judge (skipping the step above) is worse than no eval at all, because it produces a confident-looking number that may not track anything real.
- **The judge and the system under test share failure modes** — e.g. both are prone to the same factual gap, so the judge won't catch it either. Prefer a judge model from a different family/vintage when possible.

## Exercises

**Exercise 1 (Warm-up):** Change `PADDED_ANSWER` to be padded but also *subtly wrong* (state the wrong capital). Does the rubric judge still favor it over the terse-correct answer, or does correctness win out over length this time?

**Exercise 2 (Apply):** Implement `debiased_pairwise_judge(question, ans_a, ans_b)` that runs the comparison in both orders and returns `"tie"` if the two orders disagree, otherwise the consistent winner.

**Exercise 3 (Extend):** Notebook 27b builds agent-trajectory evals. Sketch which of the three biases (position, verbosity, self-preference) is most dangerous when judging an *agent's* final report rather than a single Q&A answer, and why.


In [ ]:
# Exercise 1: Warm-up
# Task: Make PADDED_ANSWER subtly incorrect and re-run rubric_judge against TERSE_ANSWER.
# Hint: keep the padding, just swap in a wrong capital city.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement debiased_pairwise_judge using both orderings of pairwise_judge.
# Hint: order 2's "A" corresponds to order 1's "B" — map back to the original answers, not slots.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Identify which judge bias is riskiest for grading an agent's final report vs a Q&A answer.
# Hint: agent reports are naturally longer and more stylistically consistent across runs of the
# same agent than two independent Q&A answers are.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
WRONG_PADDED_ANSWER = (
    "That's a great question! Let me think about this carefully. Geographically speaking, "
    "when we consider the countries of Europe and their administrative centers, the answer "
    "you're looking for, after careful consideration, is Berlin."
)
wrong_padded_rating = rubric_judge(q, WRONG_PADDED_ANSWER)
print(terse_rating, wrong_padded_rating)
# If the judge is purely verbosity-biased it may still rate this near TERSE_ANSWER despite being
# wrong — a strong signal the "penalize padding" criterion isn't explicit enough in the prompt.

# Exercise 2
def debiased_pairwise_judge(question, ans_a, ans_b):
    v1 = pairwise_judge(question, ans_a, ans_b)          # ans_a=A, ans_b=B
    v2 = pairwise_judge(question, ans_b, ans_a)           # ans_b=A, ans_a=B
    winner_1 = ans_a if v1 == "A" else ans_b
    winner_2 = ans_b if v2 == "A" else ans_a
    return winner_1 if winner_1 == winner_2 else "tie"

print(debiased_pairwise_judge("What do mitochondria do?", ANSWER_1, ANSWER_2))

# Exercise 3
# Verbosity bias is the riskiest for agent reports: a report that did MORE tool calls and
# padding naturally reads as "more thorough" to a judge, even if a shorter report that used
# fewer, better-chosen tools reached the same correct conclusion. This directly rewards the
# "agentic laziness" failure mode's opposite — agentic OVER-working — unless the judge rubric
# explicitly scores efficiency (fewer steps, lower cost) as a positive, not just correctness.
```
</details>

## Key Takeaways
- LLM judges inherit model biases plus grading-specific ones: position bias (order flips the verdict), verbosity bias (longer looks better), self-preference bias (own style rated higher).
- Pairwise comparison ranks well but is position-sensitive; rubric scoring gives trackable numbers but is verbosity-sensitive — pick based on whether you need a trend line or a final decision.
- Calibrate any judge against a small human-labeled set (aim for >0.7 correlation) before trusting it in an eval pipeline.
- Don't reach for a judge when an exact/rubric scorer already exists, when stakes are high and unverified, or before calibration.
- These same biases resurface in notebook 27b's agent-trajectory evals, where verbosity bias in particular can silently reward inefficient agents.

## What's Next
Notebook 27 puts evals into production: tracing, cost/latency monitoring, drift detection, and canary rollouts measured with the harness built across notebooks 24-26.
